In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan



# Regresion Lineal Simple

El **analisis de regresion** modela la relacion entre una variable de respuesta $y$
y una o mas variables explicativas $x$.
Es una de las herramientas estadisticas mas usadas en ingenieria:
permite cuantificar relaciones, predecir resultados y tomar decisiones basadas en datos.

En este capitulo construiremos el modelo paso a paso usando un unico caso de estudio,
y luego verificaremos cada resultado con `statsmodels`.

## Contenidos

1. **Correlacion** — medir la fuerza y direccion de la asociacion lineal.
2. **El modelo de regresion lineal simple** — supuestos y ecuacion.
3. **Bondad de ajuste** — $R^2$ y error estandar residual.
4. **Inferencia** — tests e intervalos de confianza para los coeficientes.
5. **Prediccion** — IC para la respuesta media e intervalo de prediccion.
6. **Diagnosticos** — verificar los supuestos del modelo.
7. **Que sigue** — vista previa de regresion lineal multiple.
8. **Sintesis y problemas de practica**.



---
## Caso de estudio: Concentracion de catalizador vs. Rendimiento

Un ingeniero de procesos quimicos mide como la **concentracion de catalizador** $x$ (en g/L)
afecta el **rendimiento de reaccion** $y$ (en %).
Se realizaron 20 experimentos variando la concentracion de 1 a 20 g/L.

> **Pregunta central:** existe una relacion lineal entre la concentracion del catalizador
> y el rendimiento? Cuanto aumenta el rendimiento por cada g/L adicional de catalizador?


In [ ]:
np.random.seed(2026)
concentracion = np.arange(1, 21, dtype=float)
epsilon       = np.random.normal(0, 4, 20)
rendimiento   = 30 + 3 * concentracion + epsilon

df = pd.DataFrame({"concentracion": concentracion, "rendimiento": rendimiento})

print("Dataset: Concentracion de catalizador vs. Rendimiento")
print(df.round(2).to_string(index=False))
print()
print(f"n = {len(df)}  observaciones")
print(f"Rendimiento: media = {rendimiento.mean():.2f}%,  DE = {rendimiento.std(ddof=1):.2f}%")

fig, ax = plt.subplots(figsize=(7, 5), dpi=150)
ax.scatter(concentracion, rendimiento, color="steelblue", s=50, zorder=3)
ax.set_xlabel("Concentracion de catalizador (g/L)")
ax.set_ylabel("Rendimiento de reaccion (%)")
ax.set_title("Caso de estudio: catalizador vs. rendimiento")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



---
## 1. Correlacion

Antes de ajustar cualquier modelo de regresion conviene cuantificar la
**fuerza y direccion de la asociacion lineal** entre las variables.
Si la correlacion es cercana a cero, un modelo lineal tendra poco poder predictivo.

Veremos dos medidas complementarias:
la correlacion de **Pearson** (parametrica, para relaciones lineales)
y la de **Spearman** (no parametrica, para relaciones monotonas).



### 1.1 Correlacion de Pearson

La **correlacion de Pearson** $R$ mide la fuerza de la asociacion *lineal* entre $x$ e $y$:

$$R = \frac{S_{xy}}{\sqrt{S_{xx} \cdot S_{yy}}}$$

donde:
$$S_{xy} = \sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y}), \quad
  S_{xx} = \sum_{i=1}^{n}(x_i - \bar{x})^2, \quad
  S_{yy} = \sum_{i=1}^{n}(y_i - \bar{y})^2$$

**Propiedades:** $R \in [-1,\,1]$.
Si $|R| = 1$, la relacion es perfectamente lineal.
Si $R = 0$, no hay asociacion lineal.

**Test de hipotesis** $H_0: \rho = 0$ vs. $H_a: \rho \neq 0$:

$$T_0 = \frac{R\sqrt{n-2}}{\sqrt{1-R^2}} \;\sim\; t_{n-2} \quad \text{bajo } H_0$$

> **Conexion con la regresion:** la pendiente del modelo lineal es
> $\hat{\beta}_1 = R \cdot (s_y / s_x)$.
> Por eso $R$ y $\hat{\beta}_1$ *siempre tienen el mismo signo*:
> si la correlacion es positiva, el rendimiento aumenta con la concentracion.


In [ ]:
x = df["concentracion"].values
y = df["rendimiento"].values
n = len(x)

x_bar = x.mean()
y_bar = y.mean()

S_xx = np.sum((x - x_bar) ** 2)
S_yy = np.sum((y - y_bar) ** 2)
S_xy = np.sum((x - x_bar) * (y - y_bar))

R_manual = S_xy / np.sqrt(S_xx * S_yy)
T0_corr  = R_manual * np.sqrt(n - 2) / np.sqrt(1 - R_manual**2)
p_corr   = 2 * stats.t.sf(abs(T0_corr), df=n - 2)

R_scipy, p_scipy = stats.pearsonr(x, y)

print("Calculo de la correlacion de Pearson:")
print(f"  x_bar = {x_bar:.4f},  y_bar = {y_bar:.4f}")
print(f"  S_xx  = {S_xx:.4f}")
print(f"  S_yy  = {S_yy:.4f}")
print(f"  S_xy  = {S_xy:.4f}")
print()
print(f"  R (manual) = S_xy / sqrt(S_xx * S_yy) = {R_manual:.4f}")
print(f"  R (scipy)  = {R_scipy:.4f}   (verificacion)")
print(f"  T_0        = R * sqrt(n-2) / sqrt(1-R^2) = {T0_corr:.4f}")
print(f"  p-value    = {p_corr:.4e}")
print()
if p_corr < 0.05:
    print("Conclusion: hay evidencia significativa de correlacion lineal (p < 0.05).")
else:
    print("Conclusion: no hay evidencia significativa de correlacion lineal (p >= 0.05).")


\
### 1.2 Correlacion de Spearman

La **correlacion de Spearman** ($\rho_s$) trabaja con los *rangos* de los datos
en lugar de los valores originales.
Es mas robusta frente a valores atipicos y detecta relaciones **monotonas**
(crecientes o decrecientes, no necesariamente lineales).

| | Pearson ($R$) | Spearman ($\rho_s$) |
|---|---|---|
| Mide | Relacion lineal | Relacion monotonica |
| Escala | Valores originales | Rangos |
| Sensible a outliers | Si | No |
| Supuesto distribucional | Normalidad aproximada | Ninguno |

> **Cuando usar cada una:** si los datos tienen valores extremos o la relacion
> parece curvilinea pero monotona, prefiera Spearman.
> Para datos sin outliers con relacion lineal clara, Pearson y Spearman daran resultados similares.


In [ ]:
rho_s, p_spearman = stats.spearmanr(x, y)

print("Correlacion de Spearman:")
print(f"  rho_s   = {rho_s:.4f}")
print(f"  p-value = {p_spearman:.4e}")
print()

comparacion = pd.DataFrame({
    "Metodo":  ["Pearson", "Spearman"],
    "Coef":    [round(R_scipy, 4),  round(rho_s, 4)],
    "p-value": [f"{p_scipy:.4e}", f"{p_spearman:.4e}"]
})
print(comparacion.to_string(index=False))
print()
print("Ambas correlaciones son altas y similares: la relacion es aproximadamente lineal.")



---
## 2. El Modelo de Regresion Lineal Simple

La correlacion confirmo que existe una fuerte asociacion lineal.
Ahora queremos ir mas alla: construir un **modelo cuantitativo**
que nos permita *predecir* el rendimiento dado un valor de concentracion.

El **modelo de regresion lineal simple** (SLR) describe la relacion mediante una recta:

$$Y = \beta_0 + \beta_1 x + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0,\, \sigma^2)$$

donde:

- $\beta_0$: **intercepto** — valor esperado de $Y$ cuando $x = 0$.
- $\beta_1$: **pendiente** — cambio esperado en $Y$ por una unidad de aumento en $x$.
- $\varepsilon$: **error aleatorio** — capta variabilidad que $x$ no explica.

**Los cuatro supuestos del modelo:**

1. **Linealidad:** $E[Y|x] = \beta_0 + \beta_1 x$ es lineal en $x$.
2. **Independencia:** los errores $\varepsilon_i$ son independientes entre si.
3. **Homocedasticidad:** la varianza $\text{Var}(\varepsilon) = \sigma^2$ es constante (no depende de $x$).
4. **Normalidad:** $\varepsilon \sim \mathcal{N}(0, \sigma^2)$.



### 2.1 Estimacion por Minimos Cuadrados

No conocemos los valores reales de $\beta_0$ y $\beta_1$; los estimamos
minimizando la **suma de los cuadrados de los residuos** (M\&R, ec. 11.7--11.8):

$$\min_{\beta_0,\, \beta_1} \; L = \sum_{i=1}^{n} \bigl(y_i - \beta_0 - \beta_1 x_i\bigr)^2$$

Derivando $L$ respecto a $\beta_0$ y $\beta_1$ e igualando a cero se obtienen
las **ecuaciones normales**:

$$\hat{\beta}_1 = \frac{S_{xy}}{S_{xx}}, \qquad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1\,\bar{x}$$

> **Relacion con la correlacion:** $\hat{\beta}_1 = R \cdot \dfrac{s_y}{s_x}$,
> donde $s_x$ y $s_y$ son las desviaciones estandar muestrales.
> Esto confirma que $\hat{\beta}_1$ tiene el mismo signo que $R$.


In [ ]:
beta1_hat = S_xy / S_xx
beta0_hat = y_bar - beta1_hat * x_bar

# verificacion via correlacion de Pearson
beta1_via_r = R_manual * (y.std(ddof=1) / x.std(ddof=1))

# tabla de calculo intermedio (primeras 5 filas)
tabla = pd.DataFrame({
    "x_i":          x[:5],
    "y_i":          np.round(y[:5], 2),
    "xi - xbar":    np.round((x - x_bar)[:5], 2),
    "yi - ybar":    np.round((y - y_bar)[:5], 2),
    "(xi-xb)(yi-yb)": np.round(((x - x_bar) * (y - y_bar))[:5], 2),
    "(xi-xb)^2":    np.round(((x - x_bar) ** 2)[:5], 2)
})
print("Calculo intermedio (primeras 5 observaciones):")
print(tabla.to_string(index=False))
print("  ...")
print()
print(f"Sumas sobre n = {n} observaciones:")
print(f"  x_bar = {x_bar:.4f},   y_bar = {y_bar:.4f}")
print(f"  S_xx  = {S_xx:.4f}")
print(f"  S_xy  = {S_xy:.4f}")
print()
print("Estimadores OLS (ecuaciones normales):")
print(f"  beta_1_hat = S_xy / S_xx = {S_xy:.4f} / {S_xx:.4f} = {beta1_hat:.4f}")
print(f"  beta_0_hat = y_bar - beta_1_hat*x_bar")
print(f"             = {y_bar:.4f} - {beta1_hat:.4f}*{x_bar:.1f} = {beta0_hat:.4f}")
print()
print(f"Verificacion via correlacion: beta_1 = R*(sy/sx) = {beta1_via_r:.4f}")


In [ ]:
y_hat_manual = beta0_hat + beta1_hat * x
x_range      = np.linspace(x.min() - 1, x.max() + 1, 200)
y_hat_range  = beta0_hat + beta1_hat * x_range

fig, ax = plt.subplots(figsize=(7, 5), dpi=150)
ax.scatter(x, y, color="steelblue", s=50, zorder=4, label="Datos observados")
ax.plot(x_range, y_hat_range, color="firebrick", linewidth=2,
        label=f"y_hat = {beta0_hat:.2f} + {beta1_hat:.2f}*x")

for i in range(n):
    ax.plot([x[i], x[i]], [y[i], y_hat_manual[i]],
            color="gray", linewidth=0.8, alpha=0.5)

ax.set_xlabel("Concentracion de catalizador (g/L)")
ax.set_ylabel("Rendimiento de reaccion (%)")
ax.set_title("Recta de minimos cuadrados (residuos en gris)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



### 2.2 Interpretacion de los coeficientes

> **Pendiente ($\hat{\beta}_1$):** por cada g/L adicional de catalizador,
> el rendimiento de reaccion *aumenta en promedio* $\hat{\beta}_1$ puntos porcentuales.

> **Intercepto ($\hat{\beta}_0$):** es el rendimiento esperado cuando la concentracion es 0 g/L.
> **Precaucion:** si $x = 0$ queda fuera del rango de los datos,
> el intercepto es un parametro matematico de ajuste y *no debe interpretarse fisicamente*.
> Extrapolar fuera del rango observado (1--20 g/L) es riesgoso.



### 2.3 Ajuste con statsmodels

En la practica usamos `statsmodels` para ajustar el modelo.
La funcion `summary()` entrega un reporte completo en tres bloques:

**Bloque 1 — Calidad del ajuste global:**
- `R-squared`: proporcion de varianza explicada por el modelo.
- `F-statistic` / `Prob (F-statistic)`: test de significancia global.
- `No. Observations`, `Df Residuals`: n y grados de libertad del error.

**Bloque 2 — Coeficientes:**
- `coef`: estimacion $\hat{\beta}_j$.
- `std err`: error estandar $se(\hat{\beta}_j)$.
- `t` / `P>|t|`: estadistico $T_0$ y p-value del test $H_0: \beta_j = 0$.
- `[0.025  0.975]`: intervalo de confianza al 95%.

**Bloque 3 — Diagnosticos adicionales:**
- `Omnibus` / `Jarque-Bera`: tests de normalidad de residuos.
- `Durbin-Watson`: deteccion de autocorrelacion (esperamos valor cercano a 2).

> **Nota:** `sm.add_constant(x)` agrega la columna de unos necesaria para estimar $\beta_0$.
> Sin esta llamada el modelo fuerza la recta a pasar por el origen.


In [ ]:
X      = sm.add_constant(x)
modelo = sm.OLS(y, X).fit()
print(modelo.summary())
print()

print("Valores clave extraidos del modelo:")
print(f"  beta_0_hat = {modelo.params[0]:.4f}   (intercepto)")
print(f"  beta_1_hat = {modelo.params[1]:.4f}   (pendiente)")
print(f"  R^2        = {modelo.rsquared:.4f}")
print(f"  F_0        = {modelo.fvalue:.4f},  p = {modelo.f_pvalue:.4e}")
print(f"  RSE        = {np.sqrt(modelo.mse_resid):.4f}   (error estandar residual)")
print()
print("Comparacion manual vs. statsmodels:")
print(f"  beta_0: manual = {beta0_hat:.4f},  sm = {modelo.params[0]:.4f}")
print(f"  beta_1: manual = {beta1_hat:.4f},  sm = {modelo.params[1]:.4f}")



---
## 3. Bondad de Ajuste

### Descomposicion de la varianza

La variabilidad total de $y$ se descompone en dos partes (M\&R, Tabla 11.3):

$$SS_T = SS_R + SS_E$$

| Fuente | Suma de cuadrados | gl | Cuadrado medio |
|--------|-------------------|----|----------------|
| Regresion | $SS_R = \hat{\beta}_1 S_{xy}$ | $1$ | $MS_R = SS_R / 1$ |
| Error | $SS_E = \displaystyle\sum(y_i - \hat{y}_i)^2$ | $n - 2$ | $MS_E = SS_E / (n-2)$ |
| Total | $SS_T = \displaystyle\sum(y_i - \bar{y})^2$ | $n - 1$ | |

**Coeficiente de determinacion** $R^2$:

$$R^2 = \frac{SS_R}{SS_T} = 1 - \frac{SS_E}{SS_T} \;\in [0,\,1]$$

**Error estandar residual** (RSE) — estima $\sigma$ en las mismas unidades que $y$:

$$\hat{\sigma} = RSE = \sqrt{MS_E} = \sqrt{\frac{SS_E}{n-2}}$$

> **Interpretacion:** un $R^2 = 0.95$ significa que el 95\% de la variabilidad
> del rendimiento queda explicada por la concentracion del catalizador.
> El RSE indica el error tipico de prediccion en unidades del rendimiento (puntos porcentuales).


In [ ]:
y_hat    = np.asarray(modelo.fittedvalues)
residuos = np.asarray(modelo.resid)

SS_T     = np.sum((y - y_bar) ** 2)
SS_E     = np.sum(residuos ** 2)
SS_R     = SS_T - SS_E
MS_E     = SS_E / (n - 2)
R2       = 1 - SS_E / SS_T
RSE      = np.sqrt(MS_E)
sigma_hat = RSE

print("Descomposicion de la varianza:")
print(f"  SS_T  = sum((yi - ybar)^2)     = {SS_T:.4f}  (variabilidad total)")
print(f"  SS_R  = SS_T - SS_E            = {SS_R:.4f}  (explicada por el modelo)")
print(f"  SS_E  = sum((yi - y_hat_i)^2)  = {SS_E:.4f}  (no explicada / residuos)")
print(f"  Suma de comprobacion: SS_R + SS_E = {SS_R + SS_E:.4f}  (debe ser igual a SS_T)")
print()
print(f"R^2  = 1 - SS_E/SS_T = 1 - {SS_E:.4f}/{SS_T:.4f} = {R2:.4f}")
print(f"RSE  = sqrt(SS_E/(n-2)) = sqrt({SS_E:.4f}/{n-2}) = {RSE:.4f}")
print()
print("Verificacion con statsmodels:")
print(f"  R^2 (modelo) = {modelo.rsquared:.4f}")
print(f"  RSE (modelo) = {np.sqrt(modelo.mse_resid):.4f}")
print()
print(f"Interpretacion: el modelo explica el {R2*100:.1f}% de la variabilidad del rendimiento.")
print(f"El error tipico de prediccion es {RSE:.2f} puntos porcentuales.")


---
## 4. Inferencia

### 4.1 Test para la pendiente

La pregunta mas importante: existe realmente una relacion lineal entre $x$ e $y$?

$$H_0: \beta_1 = 0 \quad \text{(no hay relacion)} \qquad H_a: \beta_1 \neq 0$$

El estadistico de prueba es (M\&R, ec. 11.24):

$$T_0 = \frac{\hat{\beta}_1}{se(\hat{\beta}_1)}, \qquad se(\hat{\beta}_1) = \sqrt{\frac{MS_E}{S_{xx}}} \qquad T_0 \sim t_{n-2} \;\text{ bajo } H_0$$

### 4.2 Intervalos de confianza para los coeficientes

IC al $100(1-\alpha)\%$ para $\beta_1$:

$$\hat{\beta}_1 \pm t_{\alpha/2,\; n-2} \cdot se(\hat{\beta}_1)$$

### 4.3 Test ANOVA para el modelo

La misma hipotesis puede evaluarse con un F-test (M\&R, Tabla 11.3):

$$F_0 = \frac{MS_R}{MS_E} \;\sim\; F_{1,\; n-2} \quad \text{bajo } H_0$$

> **Equivalencia clave:** en regresion lineal *simple* (un predictor), $F_0 = T_0^2$,
> de modo que el t-test y el F-test son equivalentes y producen el mismo p-value.
> En regresion multiple el F-test evalua la significancia *global* del modelo.


In [ ]:
se_beta1 = np.sqrt(modelo.mse_resid / S_xx)
T0_beta1 = beta1_hat / se_beta1
p_beta1  = 2 * stats.t.sf(abs(T0_beta1), df=n - 2)
t_crit   = stats.t.ppf(0.975, df=n - 2)
IC_low   = beta1_hat - t_crit * se_beta1
IC_high  = beta1_hat + t_crit * se_beta1

F0 = modelo.fvalue

print("--- Test para la pendiente (manual) ---")
print(f"  se(beta_1) = sqrt(MS_E / S_xx) = sqrt({modelo.mse_resid:.4f} / {S_xx:.2f}) = {se_beta1:.4f}")
print(f"  T_0        = beta_1_hat / se(beta_1) = {beta1_hat:.4f} / {se_beta1:.4f} = {T0_beta1:.4f}")
print(f"  t_critico  = t_{{0.025, {n-2}}} = {t_crit:.4f}")
print(f"  p-value    = {p_beta1:.4e}")
print(f"  IC 95% beta_1: [{IC_low:.4f}, {IC_high:.4f}]")
print()
print("Verificacion con statsmodels:")
ci_sm = np.asarray(modelo.conf_int())
print(f"  T_0 (modelo)   = {modelo.tvalues[1]:.4f}")
print(f"  p    (modelo)  = {modelo.pvalues[1]:.4e}")
print(f"  IC   (modelo)  = [{ci_sm[1,0]:.4f}, {ci_sm[1,1]:.4f}]")
print()
print("--- Test ANOVA ---")
print(f"  F_0        = {F0:.4f}")
print(f"  p (F-test) = {modelo.f_pvalue:.4e}")
print(f"  T_0^2      = {T0_beta1**2:.4f}   (debe coincidir con F_0)")
print()
if p_beta1 < 0.05:
    print("Conclusion: se rechaza H0. Hay evidencia significativa de que beta_1 != 0.")
    print(f"Cada g/L adicional de catalizador aumenta el rendimiento en {beta1_hat:.2f} pp")
    print(f"(IC 95%: [{IC_low:.2f}, {IC_high:.2f}] pp).")



---
## 5. Prediccion

Una vez ajustado el modelo podemos usarlo para predecir.
Existen dos tipos de prediccion con distintas incertidumbres
(M\&R, ec. 11.31 y 11.33):

### 5.1 Intervalo de confianza para la respuesta media

Estima el **valor esperado del rendimiento** para *todos* los experimentos
con concentracion $x_0$:

$$\hat{y}_0 \;\pm\; t_{\alpha/2,\, n-2} \;\cdot\; \hat{\sigma}
  \sqrt{\frac{1}{n} + \frac{(x_0 - \bar{x})^2}{S_{xx}}}$$

### 5.2 Intervalo de prediccion para una nueva observacion

Predice el rendimiento de una **observacion futura individual**
con concentracion $x_0$:

$$\hat{y}_0 \;\pm\; t_{\alpha/2,\, n-2} \;\cdot\; \hat{\sigma}
  \sqrt{1 + \frac{1}{n} + \frac{(x_0 - \bar{x})^2}{S_{xx}}}$$

> **Diferencia clave:** el intervalo de prediccion (IP) es *siempre mas ancho*
> que el IC para la respuesta media.
> El termino adicional $1$ dentro de la raiz captura la variabilidad
> de una nueva observacion individual, ademas de la incertidumbre del modelo.
> Cuanto mas alejado este $x_0$ de $\bar{x}$, mas anchos son ambos intervalos.

> **Extrapolacion:** estos intervalos suponen que el modelo es valido en $x_0$.
> Usar el modelo fuera del rango 1--20 g/L puede producir predicciones muy imprecisas.


In [ ]:
# t_crit y sigma_hat ya fueron definidos en las celdas anteriores
x0_vals    = np.linspace(x.min() - 0.5, x.max() + 0.5, 200)
y_hat_line = beta0_hat + beta1_hat * x0_vals

# IC para la respuesta media (ec. 11.31 M&R)
se_mean = sigma_hat * np.sqrt(1/n + (x0_vals - x_bar)**2 / S_xx)
ci_low  = y_hat_line - t_crit * se_mean
ci_high = y_hat_line + t_crit * se_mean

# IP para una nueva observacion (ec. 11.33 M&R)
se_pred = sigma_hat * np.sqrt(1 + 1/n + (x0_vals - x_bar)**2 / S_xx)
pi_low  = y_hat_line - t_crit * se_pred
pi_high = y_hat_line + t_crit * se_pred

# Ejemplo puntual: x_0 = 10 g/L
x0_ej    = 10.0
y_hat_ej = beta0_hat + beta1_hat * x0_ej
se_m_ej  = sigma_hat * np.sqrt(1/n + (x0_ej - x_bar)**2 / S_xx)
se_p_ej  = sigma_hat * np.sqrt(1 + 1/n + (x0_ej - x_bar)**2 / S_xx)
ci_l_ej  = y_hat_ej - t_crit * se_m_ej
ci_u_ej  = y_hat_ej + t_crit * se_m_ej
pi_l_ej  = y_hat_ej - t_crit * se_p_ej
pi_u_ej  = y_hat_ej + t_crit * se_p_ej

print(f"Prediccion para x_0 = {x0_ej} g/L:")
print(f"  y_hat_0                  = {y_hat_ej:.4f} %")
print(f"  SE respuesta media       = {se_m_ej:.4f}")
print(f"  SE prediccion nueva      = {se_p_ej:.4f}  (+1 bajo la raiz)")
print()
print(f"  IC 95% respuesta media:   [{ci_l_ej:.4f}, {ci_u_ej:.4f}] %")
print(f"  IP 95% observacion nueva: [{pi_l_ej:.4f}, {pi_u_ej:.4f}] %")
print()
print(f"  Amplitud IC: {ci_u_ej - ci_l_ej:.4f} pp")
print(f"  Amplitud IP: {pi_u_ej - pi_l_ej:.4f} pp  (siempre mayor que IC)")

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
ax.scatter(x, y, color="steelblue", s=50, zorder=4, label="Datos observados")
ax.plot(x0_vals, y_hat_line, color="firebrick", linewidth=2, label="Recta ajustada")
ax.fill_between(x0_vals, ci_low, ci_high,
                alpha=0.3, color="firebrick", label="IC 95% respuesta media")
ax.fill_between(x0_vals, pi_low, pi_high,
                alpha=0.15, color="orange", label="IP 95% obs. nueva")
ax.axvline(x0_ej, color="gray", linestyle=":", alpha=0.7)
ax.set_xlabel("Concentracion de catalizador (g/L)")
ax.set_ylabel("Rendimiento de reaccion (%)")
ax.set_title("Intervalos de confianza y prediccion")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



---
## 6. Diagnosticos del Modelo

Ajustar una recta no garantiza que los supuestos del modelo se cumplan.
Los **diagnosticos** son pasos obligatorios antes de interpretar o usar el modelo.

| Supuesto | Grafico recomendado | Test formal |
|----------|---------------------|-------------|
| Linealidad | Residuos vs $\hat{y}$ | — |
| Homocedasticidad | Residuos vs $\hat{y}$ | Breusch-Pagan |
| Normalidad de errores | Q-Q plot | Shapiro-Wilk |
| Independencia | Residuos vs orden temporal | Durbin-Watson |
| Puntos influyentes | Distancia de Cook | $D_i > 4/n$ |

**Residuo:** $\hat{\varepsilon}_i = y_i - \hat{y}_i$.
Si el modelo es correcto, los residuos deben parecerse a una muestra
de $\mathcal{N}(0, \sigma^2)$, dispersa aleatoriamente sin patrones.


In [ ]:
influence    = modelo.get_influence()
cook_d, _    = influence.cooks_distance
umbral_cook  = 4 / n

fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=150)

# Panel 1: residuos vs valores predichos
axes[0].scatter(y_hat, residuos, color="steelblue", s=40, zorder=3)
axes[0].axhline(0, color="firebrick", linestyle="--")
axes[0].set_xlabel("Valores predichos")
axes[0].set_ylabel("Residuos")
axes[0].set_title("Residuos vs. Predichos")
axes[0].grid(alpha=0.3)

# Panel 2: Q-Q plot
(osm, osr), (slope, intercept, _) = stats.probplot(residuos, dist="norm")
axes[1].scatter(osm, osr, color="steelblue", s=40)
axes[1].plot(osm, slope * osm + intercept, color="firebrick")
axes[1].set_xlabel("Cuantiles teoricos")
axes[1].set_ylabel("Cuantiles observados")
axes[1].set_title("Q-Q plot de residuos")
axes[1].grid(alpha=0.3)

# Panel 3: distancia de Cook
colors_cook = ["firebrick" if d > umbral_cook else "steelblue" for d in cook_d]
axes[2].bar(range(n), cook_d, color=colors_cook, alpha=0.7)
axes[2].axhline(umbral_cook, color="firebrick", linestyle="--",
                label=f"Umbral = 4/n = {umbral_cook:.3f}")
axes[2].set_xlabel("Indice de observacion")
axes[2].set_ylabel("Distancia de Cook")
axes[2].set_title("Distancia de Cook")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle("Diagnosticos del modelo", fontsize=12)
plt.tight_layout()
plt.show()

influyentes = np.where(cook_d > umbral_cook)[0]
if len(influyentes):
    print(f"Observaciones influyentes (D_i > {umbral_cook:.3f}): indices {influyentes}")
    print(df.iloc[influyentes].round(2).to_string(index=False))
else:
    print(f"Ninguna observacion supera el umbral D_i > {umbral_cook:.3f}.")


In [ ]:
sw_stat, sw_p = stats.shapiro(residuos)
_, bp_p, _, _ = het_breuschpagan(residuos, modelo.model.exog)

print("Tests formales de supuestos:")
print()
print(f"  Normalidad   (Shapiro-Wilk):   W = {sw_stat:.4f},  p = {sw_p:.4f}")
print(f"  Homocedasticidad (Breusch-Pagan):       p = {bp_p:.4f}")
print()

norm_ok = sw_p  > 0.05
homo_ok = bp_p  > 0.05

print("Conclusion:")
print(f"  Normalidad:       {'OK  (p > 0.05, no se rechaza H0)' if norm_ok else 'VIOLADA (p <= 0.05, se rechaza H0)'}")
print(f"  Homocedasticidad: {'OK  (p > 0.05, no se rechaza H0)' if homo_ok else 'VIOLADA (p <= 0.05, se rechaza H0)'}")



> **Como interpretar los graficos de diagnostico:**
>
> - **Residuos vs. Predichos:** los puntos deben dispersarse *aleatoriamente* alrededor de cero,
>   sin patrones en forma de abanico (heterocedasticidad) ni curva (no linealidad).
> - **Q-Q plot:** los puntos deben seguir la linea roja diagonal.
>   Desviaciones en los extremos indican colas pesadas; una curva sistematica indica asimetria.
> - **Distancia de Cook:** puntos por encima del umbral $4/n$ tienen alta influencia
>   sobre los coeficientes estimados. Vale la pena investigarlos
>   (posibles errores de medicion u observaciones atipicas).
>
> Si alguno de los supuestos se viola, los p-values e intervalos de confianza
> pueden no ser confiables. Considere transformaciones de variables o modelos mas robustos.



---
## 7. Que sigue: Regresion Lineal Multiple

En muchos procesos reales el rendimiento depende de varios factores simultaneamente.
La **regresion lineal multiple** (MLR) extiende el modelo SLR a $p$ predictores:

$$Y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p + \varepsilon$$

En notacion matricial: $\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon}$,
con estimador OLS $\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$.

Los temas que abordaremos la proxima sesion son:

- **$R^2$ ajustado:** penaliza el numero de predictores para evitar el sobreajuste.
- **Seleccion de variables:** como decidir que predictores incluir.
- **Multicolinealidad:** cuando los predictores estan correlacionados entre si,
  los coeficientes se vuelven inestables.
  El **Factor de Inflacion de la Varianza (VIF)** permite detectarlo.
- **Diagnosticos para MLR:** interpretacion del F-test global vs. t-tests individuales.

> *Proxima sesion: Regresion Lineal Multiple (M\&R, Capitulo 12)*



---
## 8. Sintesis

### Flujo de trabajo recomendado

1. **Exploracion:** scatter plot + correlacion de Pearson y Spearman.
2. **Ajuste:** `sm.OLS(y, sm.add_constant(x)).fit()`.
3. **Bondad de ajuste:** $R^2$ (varianza explicada) y RSE (error en unidades de $y$).
4. **Inferencia:** t-test para $\hat{\beta}_1$, IC al 95%, F-test global ($F_0 = T_0^2$ en SLR).
5. **Prediccion:** IC para la respuesta media (ec. 11.31) vs. IP para obs. nueva (ec. 11.33).
6. **Diagnosticos:** Q-Q plot (normalidad), residuos vs. $\hat{y}$ (homocedasticidad),
   Cook's distance (influencia), Shapiro-Wilk, Breusch-Pagan.

### Tabla de referencia rapida

| Concepto | Formula / Definicion | Python |
|---|---|---|
| Pearson | $R = S_{xy}/\sqrt{S_{xx} S_{yy}}$ | `stats.pearsonr(x, y)` |
| Spearman | Por rangos | `stats.spearmanr(x, y)` |
| OLS | $\hat\beta_1 = S_{xy}/S_{xx}$, $\hat\beta_0 = \bar y - \hat\beta_1 \bar x$ | `sm.OLS(y, sm.add_constant(x)).fit()` |
| $R^2$ | $1 - SS_E/SS_T$ | `model.rsquared` |
| RSE | $\sqrt{SS_E/(n-2)}$ | `np.sqrt(model.mse_resid)` |
| Test pendiente | $T_0 = \hat\beta_1 / se(\hat\beta_1) \sim t_{n-2}$ | `model.tvalues`, `model.pvalues` |
| IC coeficientes | $\hat\beta_1 \pm t_{\alpha/2,n-2} \cdot se(\hat\beta_1)$ | `model.conf_int()` |
| F-test | $F_0 = MS_R/MS_E \sim F_{1,n-2}$; $F_0 = T_0^2$ en SLR | `model.fvalue` |
| IC respuesta media | $\hat y_0 \pm t\,\hat\sigma\sqrt{1/n + (x_0-\bar x)^2/S_{xx}}$ | ec. 11.31 M\&R |
| IP obs. nueva | $\hat y_0 \pm t\,\hat\sigma\sqrt{1 + 1/n + (x_0-\bar x)^2/S_{xx}}$ | ec. 11.33 M\&R |
| Normalidad | Q-Q plot, Shapiro-Wilk | `stats.shapiro(resid)` |
| Homocedasticidad | Residuos vs $\hat y$, Breusch-Pagan | `het_breuschpagan(resid, exog)` |
| Influencia | Distancia de Cook $D_i > 4/n$ | `model.get_influence().cooks_distance` |

**Referencia:** Montgomery & Runger (2019), *Applied Statistics and Probability for Engineers*,
7a ed., Wiley. Capitulos 11--12.



---
## Problemas de Practica

Los siguientes problemas usan el **dataset clasico de pureza de oxigeno**
de Montgomery & Runger (2019), Capitulo 11.
Un ingeniero estudia como el nivel de hidrocarburos $x$ (%) en el gas de entrada
afecta la pureza del oxigeno producido $y$ (%).

Reproduzca el analisis paso a paso para afianzar los conceptos del capitulo.


In [ ]:
data_ox = {
    "hc":     [0.99, 1.02, 1.15, 1.29, 1.46, 1.36, 0.87, 1.23, 1.55, 1.40,
               1.19, 1.15, 0.98, 1.01, 1.11, 1.20, 1.26, 1.32, 1.43, 0.95],
    "pureza": [90.01, 89.05, 91.43, 93.74, 96.73, 94.45, 87.59, 91.77,
               99.42, 93.65, 93.54, 92.52, 90.56, 89.54, 89.85, 90.39,
               93.25, 93.41, 94.98, 87.33]
}
df_ox = pd.DataFrame(data_ox)

print("Dataset de pureza de oxigeno (Montgomery & Runger, 2019, Capitulo 11):")
print(df_ox.to_string(index=False))

# --- Paso 1: Calcule la correlacion de Pearson y el test de hipotesis.
#     Es significativa?

# --- Paso 2: Estime beta_0 y beta_1 con OLS manual (usando S_xy y S_xx).
#     Compare con sm.OLS.

# --- Paso 3: Calcule SS_T, SS_R, SS_E, R^2 y RSE.

# --- Paso 4: Para x_0 = 1.25% de hidrocarburos, calcule:
#     (a) IC 95% para la respuesta media.
#     (b) IP 95% para una nueva observacion.

# --- Paso 5: Grafique la recta ajustada con IC y IP.

# --- Paso 6: Verifique los supuestos con Q-Q plot, residuos vs y_hat
#     y distancia de Cook. Corra Shapiro-Wilk y Breusch-Pagan.
#     Se cumplen los supuestos?



### Problema 2: Interpretacion conceptual

Responda las siguientes preguntas *sin correr codigo*:

1. En el modelo ajustado para la pureza del oxigeno,
   que representa el intercepto $\hat{\beta}_0$?
   Tiene sentido fisico en este contexto?
   Por que si o por que no?

2. Explique en sus propias palabras por que el intervalo de prediccion
   es siempre mas ancho que el intervalo de confianza para la respuesta media.
   Use las formulas del capitulo para justificar su respuesta.

3. Un colega propone predecir la pureza del oxigeno para un nivel de hidrocarburos
   de $x_0 = 3\%$, muy por encima del rango observado (0.87--1.55%).
   Que problemas tiene esta extrapolacion?
   Que le recomendaria a su colega?
